# GT match 제출 이후 —

`data/interim/gt_match_results.csv`(4.0 결과)를 입력으로 사용. 

In [19]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src" / "utils") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src" / "utils"))

REPO_ROOT

WindowsPath('d:/Study/dongguk_university/dreampath')

In [20]:
import pandas as pd

df = pd.read_csv(
    REPO_ROOT / "data" / "interim" / "gt_match_results.csv", encoding="utf-8-sig"
)
df["school_candidate"] = df["school_candidate"].fillna("")
df["preprocessed_candidate"] = df["preprocessed_candidate"].fillna("")
df["ans"] = df["ans"].fillna("")
df.shape

(1000, 11)

In [21]:
gt_df = pd.read_csv(REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv", encoding="utf-8-sig")
gt_names = set(gt_df["학교명"])

alias_to_canonical = {}
if "약어" in gt_df.columns:
    for aliases, name in zip(gt_df["약어"], gt_df["학교명"]):
        if pd.notna(aliases):
            for alias in aliases.split():
                alias_to_canonical[alias] = name

In [22]:
df.columns

Index(['comment_id', 'comment', 'comment_noun', 'school_candidate',
       'preprocessed_candidate', 'school_candidate_count', 'gt_match',
       'gt_match_count', 'ans', 'ans_count', 'comment_clean_1'],
      dtype='str')

In [23]:
# 4.0의 TOKEN_FIXES/JUNK_TOKENS 보정 이후에도 여전히 ans_count == 0으로 남은 행
still_zero_df = df[df["ans_count"] == 0]
print(f"ans_count==0인 행 {len(still_zero_df)}개")
still_zero_df[["comment_id", "comment_clean_1", "school_candidate", "preprocessed_candidate"]]

ans_count==0인 행 16개


,comment_id,comment_clean_1,school_candidate,preprocessed_candidate
133,C0134,잠 실 중 치킨 부탁드려요,중,중학교
140,C0141,배민 최고 서울고도 꼭 뽑아주세요,최고,최고등학교
388,C0389,반 포 중 치킨 가즈아,중,중학교
436,C0437,휘 문 고 치킨 가즈아,고,고등학교
443,C0444,배민 최고 잠실중도 꼭 뽑아주세요,최고,최고등학교
559,C0560,배민 최고 대치초도 꼭 뽑아주세요,최고,최고등학교
576,C0577,배민 최고 대치중도 꼭 뽑아주세요,최고,최고등학교
610,C0611,휘 문 고 배민이벤트,고,고등학교
689,C0690,배민 최고 서초초도 꼭 뽑아주세요,최고,최고등학교
695,C0696,배민 최고 잠실중도 꼭 뽑아주세요,최고,최고등학교


케이스:
1. 학교 — GT match 전 0건으로 안 잡혀서 GT match 로직을 못 따라간 경우
2. 띄어쓰기 — "서 울 고" 등 (기존 `merge_spaced_syllables` 재사용)
3. "최고" 같은 오검출 단어 — 건너뛰기 (기존 `JUNK_TOKENS`가 이미 처리)
4. `[ ]` 대괄호 — 전처리 안 되어 있던 부분, 처리 후 1번 케이스로 흡수

## 1. 0건인 경우 

1. [] 없애고 
2. 띄어쓰기 붙여주고 (기존 로직 사용)
3. 기존 전처리 로직 돌리기 ( 해당 건들은 1건으로 이미 잡혀서 4.0에서 진행한 전처리 로직들을 못 따라 오류가 난 상황 다시 돌리기만 해도 ans 채우기 가능)

#### 1) 대괄호 삭제

In [24]:
import re


def strip_local_brackets(text: str) -> str:
    """대괄호/중괄호/소괄호 문자만 지역적으로 제거 (안의 내용은 유지).
    preprocessing.py의 clean_text는 안 건드리고 이 노트북 안에서만 처리."""
    return re.sub(r"[\[\]{}()]", " ", text)


still_zero_df = still_zero_df.copy()
still_zero_df["comment_clean_2"] = still_zero_df["comment_clean_1"].apply(strip_local_brackets)
still_zero_df[["comment_id", "comment_clean_1", "comment_clean_2"]]

,comment_id,comment_clean_1,comment_clean_2
133,C0134,잠 실 중 치킨 부탁드려요,잠 실 중 치킨 부탁드려요
140,C0141,배민 최고 서울고도 꼭 뽑아주세요,배민 최고 서울고도 꼭 뽑아주세요
388,C0389,반 포 중 치킨 가즈아,반 포 중 치킨 가즈아
436,C0437,휘 문 고 치킨 가즈아,휘 문 고 치킨 가즈아
443,C0444,배민 최고 잠실중도 꼭 뽑아주세요,배민 최고 잠실중도 꼭 뽑아주세요
559,C0560,배민 최고 대치초도 꼭 뽑아주세요,배민 최고 대치초도 꼭 뽑아주세요
576,C0577,배민 최고 대치중도 꼭 뽑아주세요,배민 최고 대치중도 꼭 뽑아주세요
610,C0611,휘 문 고 배민이벤트,휘 문 고 배민이벤트
689,C0690,배민 최고 서초초도 꼭 뽑아주세요,배민 최고 서초초도 꼭 뽑아주세요
695,C0696,배민 최고 잠실중도 꼭 뽑아주세요,배민 최고 잠실중도 꼭 뽑아주세요


#### 2) 띄어쓴 음절 병합

In [25]:
from preprocessing import merge_spaced_syllables, extract_school_candidates_fallback

merged = still_zero_df["comment_clean_2"].apply(merge_spaced_syllables)
new_school_candidate = merged.apply(extract_school_candidates_fallback)

still_zero_df["school_candidate"] = new_school_candidate
df.loc[still_zero_df.index, "school_candidate"] = new_school_candidate

In [26]:
still_zero_df[["comment_id", "comment_clean_2", "school_candidate"]]

,comment_id,comment_clean_2,school_candidate
133,C0134,잠 실 중 치킨 부탁드려요,잠실중
140,C0141,배민 최고 서울고도 꼭 뽑아주세요,최고 서울고
388,C0389,반 포 중 치킨 가즈아,반포중
436,C0437,휘 문 고 치킨 가즈아,휘문고
443,C0444,배민 최고 잠실중도 꼭 뽑아주세요,최고 잠실중
559,C0560,배민 최고 대치초도 꼭 뽑아주세요,최고 대치초
576,C0577,배민 최고 대치중도 꼭 뽑아주세요,최고 대치중
610,C0611,휘 문 고 배민이벤트,휘문고
689,C0690,배민 최고 서초초도 꼭 뽑아주세요,최고 서초
695,C0696,배민 최고 잠실중도 꼭 뽑아주세요,최고 잠실중


#### 3) ans 뽑아내기

In [27]:
from utils import preprocess_candidate, gt_match, apply_token_fixes

# 4.0과 동일하게 재정의
TOKEN_FIXES = {
    "서초등학교": "서초초등학교",
    "인하대부속초등학교": "인하대학교부속초등학교",
}
JUNK_TOKENS = {"학교", "고등학교", "중학교", "최고등학교", "초등학교"}

new_preprocessed = still_zero_df["school_candidate"].apply(
    lambda s: preprocess_candidate(s, alias_to_canonical)
)
new_gt_match = new_preprocessed.apply(lambda s: gt_match(s, gt_names))
new_gt_match_count = new_gt_match.apply(lambda s: len(s.split()) if s else 0)
new_ans = new_preprocessed.apply(lambda s: apply_token_fixes(s, TOKEN_FIXES, JUNK_TOKENS))
new_ans_count = new_ans.apply(lambda s: len(s.split()) if s else 0)

still_zero_df["preprocessed_candidate"] = new_preprocessed
still_zero_df["ans"] = new_ans
still_zero_df["ans_count"] = new_ans_count

df.loc[still_zero_df.index, "preprocessed_candidate"] = new_preprocessed
df.loc[still_zero_df.index, "gt_match"] = new_gt_match
df.loc[still_zero_df.index, "gt_match_count"] = new_gt_match_count
df.loc[still_zero_df.index, "ans"] = new_ans
df.loc[still_zero_df.index, "ans_count"] = new_ans_count

still_zero_df[["comment_id", "school_candidate", "preprocessed_candidate", "ans", "ans_count"]]

,comment_id,school_candidate,preprocessed_candidate,ans,ans_count
133,C0134,잠실중,잠실중학교,잠실중학교,1
140,C0141,최고 서울고,최고등학교 서울고등학교,서울고등학교,1
388,C0389,반포중,반포중학교,반포중학교,1
436,C0437,휘문고,휘문고등학교,휘문고등학교,1
443,C0444,최고 잠실중,최고등학교 잠실중학교,잠실중학교,1
559,C0560,최고 대치초,최고등학교 대치초등학교,대치초등학교,1
576,C0577,최고 대치중,최고등학교 대치중학교,대치중학교,1
610,C0611,휘문고,휘문고등학교,휘문고등학교,1
689,C0690,최고 서초,최고등학교 서초등학교,서초초등학교,1
695,C0696,최고 잠실중,최고등학교 잠실중학교,잠실중학교,1


In [28]:
# 띄어쓰기 병합 + ans 재추출 반영 후에도 여전히 ans_count == 0으로 남은 행 확인
still_zero_df_3 = df[df["ans_count"] == 0]
print(f"반영 후에도 ans_count==0인 행 {len(still_zero_df_3)}개")
still_zero_df_3[["comment_id", "comment", "school_candidate", "ans"]]

반영 후에도 ans_count==0인 행 0개


,comment_id,comment,school_candidate,ans


In [29]:
df.columns

Index(['comment_id', 'comment', 'comment_noun', 'school_candidate',
       'preprocessed_candidate', 'school_candidate_count', 'gt_match',
       'gt_match_count', 'ans', 'ans_count', 'comment_clean_1'],
      dtype='str')

## 2. 2건인 경우 보정하기

In [30]:
# 보정 후에도 여전히 ans_count == 0으로 남은 행이 있는지 확인
still_zero_df_2 = df[df["ans_count"] == 2]
print(f"2건인 경우 확인하기 {len(still_zero_df_2)}개")
still_zero_df_2[["comment_id", "comment_clean_1", "school_candidate", "ans"]]

2건인 경우 확인하기 49개


,comment_id,comment_clean_1,school_candidate,ans
3,C0004,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중 건국대,잠실중학교 건국대학교
32,C0033,중앙대 친구가 놀러왔지만 치킨은 인하대로 주세요,중앙대 인하대,중앙대학교 인하대학교
49,C0050,인하대 친구가 놀러왔지만 치킨은 대치중로 주세요,인하대 대치중,인하대학교 대치중학교
88,C0089,잠원초 친구랑 인하부중 친구 같이 보고 있는데 저는 잠원초로 신청,잠원초 인하부중 보고,잠원초등학교 인하대학교사범대학부속중학교
123,C0124,인하대 친구랑 대치중 친구 같이 보고 있는데 저는 인하대로 신청,인하대 대치중 보고,인하대학교 대치중학교
127,C0128,인하부중 친구랑 건대 친구 같이 보고 있는데 저는 인하부중로 신청,인하부중 건대 보고,인하대학교사범대학부속중학교 건국대학교
148,C0149,대치중 동아리랑 인하대부속중 동아리 같이 응원하지만 대표는 대치중,대치중 인하대부속중,대치중학교 인하대학교사범대학부속중학교
165,C0166,한양대 vs 성대 얘기하다가 결국 한양대로 참여합니다,한양대 성대,한양대학교 성균관대학교
168,C0169,인하대부속고 동아리랑 서초중 동아리 같이 응원하지만 대표는 인하대부속고,인하대부속고 서초중,인하대학교사범대학부속고등학교 서초중학교
179,C0180,연대 vs 휘문고 얘기하다가 결국 연대로 참여합니다,연대 휘문고,연세대학교 휘문고등학교


해당건들은 문맥을 파악해야한다 
- 만약 데이터가 많아진다면 llm api를 이용하여 분류해야한다. (이때 llm 비결정성 문제를 고려하여 llm은 T,F만 판단하게 한다. 2개의 고등학교를 각각 열로 만들고 llm이 판단했을때 넣어야하는 학교에 t를 적재하게 만든다.)
- 하지만 49건밖에 안됨으로 읽어보며 규칙 파악 
    => 문장 뒤에 나온 학교가 진짜 신청하는 학교로 파악됨 
    - 하지만 실제 comment에서의 순서랑 ans의 순서가 달라 재편성한 이후 진행 

### 1) 순서 통일

In [31]:
two_df = df[df["ans_count"] == 2].copy()

# school_candidate에서 "보고" 같은 오검출 토큰 제거 (49건 전체에서 유일하게 섞이는 토큰 —
# "~친구랑 ~친구 같이 보고 있는데 저는 ~로 신청" 패턴에서 "보고 있는데"의 "보고"가 걸림)
two_df["school_candidate_clean"] = two_df["school_candidate"].apply(
    lambda s: " ".join(tok for tok in s.split() if tok != "보고")
)


def order_by_last_position(row):
    """school_candidate_clean과 ans는 같은 파이프라인에서 순서 유지되며 만들어졌기 때문에
    인덱스로 1:1 대응됨. 다만 그 리스트 순서(=원문 첫 등장 순서)가 우리가 원하는 순서는 아니므로,
    각 축약형을 comment_clean_1에서 rfind로 다시 검색해 실제 마지막 등장 위치 기준으로 재정렬."""
    candidates = row["school_candidate_clean"].split()
    canonicals = row["ans"].split()
    comment = row["comment_clean_1"]
    scored = [(comment.rfind(cand), canon) for cand, canon in zip(candidates, canonicals)]
    scored.sort(key=lambda x: x[0])
    return " ".join(canon for _, canon in scored)


two_df["ans_order"] = two_df.apply(order_by_last_position, axis=1)
two_df[["comment_id", "comment_clean_1", "school_candidate_clean", "ans", "ans_order"]]

,comment_id,comment_clean_1,school_candidate_clean,ans,ans_order
3,C0004,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중 건국대,잠실중학교 건국대학교,잠실중학교 건국대학교
32,C0033,중앙대 친구가 놀러왔지만 치킨은 인하대로 주세요,중앙대 인하대,중앙대학교 인하대학교,중앙대학교 인하대학교
49,C0050,인하대 친구가 놀러왔지만 치킨은 대치중로 주세요,인하대 대치중,인하대학교 대치중학교,인하대학교 대치중학교
88,C0089,잠원초 친구랑 인하부중 친구 같이 보고 있는데 저는 잠원초로 신청,잠원초 인하부중,잠원초등학교 인하대학교사범대학부속중학교,인하대학교사범대학부속중학교 잠원초등학교
123,C0124,인하대 친구랑 대치중 친구 같이 보고 있는데 저는 인하대로 신청,인하대 대치중,인하대학교 대치중학교,대치중학교 인하대학교
127,C0128,인하부중 친구랑 건대 친구 같이 보고 있는데 저는 인하부중로 신청,인하부중 건대,인하대학교사범대학부속중학교 건국대학교,건국대학교 인하대학교사범대학부속중학교
148,C0149,대치중 동아리랑 인하대부속중 동아리 같이 응원하지만 대표는 대치중,대치중 인하대부속중,대치중학교 인하대학교사범대학부속중학교,인하대학교사범대학부속중학교 대치중학교
165,C0166,한양대 vs 성대 얘기하다가 결국 한양대로 참여합니다,한양대 성대,한양대학교 성균관대학교,성균관대학교 한양대학교
168,C0169,인하대부속고 동아리랑 서초중 동아리 같이 응원하지만 대표는 인하대부속고,인하대부속고 서초중,인하대학교사범대학부속고등학교 서초중학교,서초중학교 인하대학교사범대학부속고등학교
179,C0180,연대 vs 휘문고 얘기하다가 결국 연대로 참여합니다,연대 휘문고,연세대학교 휘문고등학교,휘문고등학교 연세대학교


### 2) ans 돌리기

In [32]:
# ans_order에서 맨 뒤(=원문에서 가장 나중에 등장)에 있는 학교를 최종 정답으로 채택
fin_ans = two_df["ans_order"].apply(lambda s: s.split()[-1])

df.loc[two_df.index, "ans"] = fin_ans
df.loc[two_df.index, "ans_count"] = df.loc[two_df.index, "ans"].apply(
    lambda s: len(s.split()) if s else 0
)

df.loc[two_df.index, ["comment_id", "comment", "ans", "ans_count"]]

,comment_id,comment,ans,ans_count
3,C0004,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,건국대학교,1
32,C0033,중앙대 친구가 놀러왔지만 치킨은 인하대로 주세요,인하대학교,1
49,C0050,인하대 친구가 놀러왔지만 치킨은 대치중로 주세요,대치중학교,1
88,C0089,잠원초 친구랑 인하부중 친구 같이 보고 있는데 저는 잠원초로 신청!,잠원초등학교,1
123,C0124,인하대 친구랑 대치중 친구 같이 보고 있는데 저는 인하대로 신청!,인하대학교,1
127,C0128,인하부중 친구랑 건대 친구 같이 보고 있는데 저는 인하부중로 신청!,인하대학교사범대학부속중학교,1
148,C0149,대치중 동아리랑 인하대부속중 동아리 같이 응원하지만 대표는 대치중,대치중학교,1
165,C0166,한양대 vs 성대 얘기하다가 결국 한양대로 참여합니다,한양대학교,1
168,C0169,인하대부속고 동아리랑 서초중 동아리 같이 응원하지만 대표는 인하대부속고,인하대학교사범대학부속고등학교,1
179,C0180,연대 vs 휘문고 얘기하다가 결국 연대로 참여합니다,연세대학교,1


In [33]:
df["ans_count"].value_counts()

ans_count
1    1000
Name: count, dtype: int64

In [34]:
df.columns

Index(['comment_id', 'comment', 'comment_noun', 'school_candidate',
       'preprocessed_candidate', 'school_candidate_count', 'gt_match',
       'gt_match_count', 'ans', 'ans_count', 'comment_clean_1'],
      dtype='str')

## 3. 마지막 체크 

In [35]:
from utils import is_structurally_valid_school_name

# ans에 구조 검증(접두 어절 >= 2글자)을 통과 못하는 토큰이 섞인 행 확인
flagged = df[
    df["ans"].apply(lambda s: any(not is_structurally_valid_school_name(tok) for tok in s.split()))
]
print(f"구조 검증 실패로 걸러질 행 {len(flagged)}개")
flagged[["comment_id", "comment_clean_1", "school_candidate", "ans"]]

구조 검증 실패로 걸러질 행 5개


,comment_id,comment_clean_1,school_candidate,ans
228,C0229,인 하 부 초 치킨 부탁드려요,초,초등학교
245,C0246,반 포 초 오늘만 기다렸어요,초,초등학교
372,C0373,서 초 초 치킨 제발,초,초등학교
609,C0610,서 초 중 배민이벤트,초 중,초등학교
705,C0706,이 대 부 초 치킨 제발,초,초등학교


원인 : 실제 gt에 "초등학교"라는 잘못된 데이터 적재되어 있음 
- gt라고 전부 믿을 수 없다. 다음부터는 gt에서도 검증 로직을 넣어야한다.

#### 1) ans 변경

In [36]:
from preprocessing import merge_spaced_syllables, extract_school_candidates_longest
from utils import preprocess_candidate, apply_token_fixes

# flagged(구조 검증 실패 목록, 6edb1139)를 직접 기준으로 씀 — ans_count==0을 기준으로 하면
# 중간에 토큰 제거 셀을 안 거치고 처음부터 다시 실행했을 때 이 행들을 못 찾음
JUNK_TOKENS = {"학교", "고등학교", "중학교", "최고등학교", "초등학교"}

# extract_school_candidates_fallback(가장 짧은 매칭)이 아니라 extract_school_candidates_longest
# (가장 긴 매칭)를 씀 — "서초중"/"이대부초"처럼 짧은 접미사("서초"/"이대")가 먼저 걸려서
# 뒤에 이어지는 글자를 놓치는 문제 때문. longest 매칭을 쓰면 애초에 통째로 뽑혀서
# TOKEN_FIXES(서초등학교->서초초등학교 등)가 필요했던 케이스 자체가 안 생김
merged = flagged["comment_clean_1"].apply(merge_spaced_syllables)
new_school_candidate = merged.apply(extract_school_candidates_longest)
new_preprocessed = new_school_candidate.apply(lambda s: preprocess_candidate(s, alias_to_canonical))
new_ans = new_preprocessed.apply(lambda s: apply_token_fixes(s, {}, JUNK_TOKENS))

df.loc[flagged.index, "school_candidate"] = new_school_candidate
df.loc[flagged.index, "preprocessed_candidate"] = new_preprocessed
df.loc[flagged.index, "ans"] = new_ans
df.loc[flagged.index, "ans_count"] = new_ans.apply(lambda s: len(s.split()) if s else 0)

df.loc[flagged.index, ["comment_id", "comment", "school_candidate", "ans", "ans_count"]]

,comment_id,comment,school_candidate,ans,ans_count
228,C0229,인 하 부 초!!! 치킨 부탁드려요,인하부초,인하부초등학교,1
245,C0246,반 포 초.. 오늘만 기다렸어요,반포초,반포초등학교,1
372,C0373,서 초 초 / 치킨 / 제발,서초초,서초초등학교,1
609,C0610,#서 초 중 #배민이벤트,서초중,서초중학교,1
705,C0706,이 대 부 초 / 치킨 / 제발,이대부초,이화여자대학교사범대학부속초등학교,1


#### 2) 찐 마지막 체크

In [38]:
from utils import is_structurally_valid_school_name

# ans에 구조 검증(접두 어절 >= 2글자)을 통과 못하는 토큰이 섞인 행 확인
flagged = df[
    df["ans"].apply(lambda s: any(not is_structurally_valid_school_name(tok) for tok in s.split()))
]
print(f"구조 검증 실패로 걸러질 행 {len(flagged)}개")
flagged[["comment_id", "comment_clean_1", "school_candidate", "ans"]]

구조 검증 실패로 걸러질 행 0개


,comment_id,comment_clean_1,school_candidate,ans


## 4. fin 결과 만들기

In [39]:
out_path = REPO_ROOT / "data" / "processed" / "fin_result.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df[
    [
        "comment_id",
        "comment",
        "comment_noun",
        "school_candidate",
        "preprocessed_candidate",
        "school_candidate_count",
        "gt_match",
        "gt_match_count",
        "ans",
        "ans_count",
    ]
].to_csv(out_path, index=False, encoding="utf-8-sig")
out_path

WindowsPath('d:/Study/dongguk_university/dreampath/data/processed/fin_result.csv')